# ULMFiT — Universal Language Model Fine-tuning for Text Classification (2018)

_Papers · Sequence & NLP_

**Pretrain a language model on raw text, then fine-tune it for classification with three tricks — ImageNet-style transfer learning, finally working for NLP.**

---

This notebook is a practice scaffold for the **ULMFiT — Universal Language Model Fine-tuning for Text Classification (2018)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, numpy as np, math
torch.manual_seed(0); np.random.seed(0)

# --- 0a. Worked example: STLR schedule (Eqn 3). T=1000, cut_frac=.1, ratio=32, eta_max=.01 ---
def stlr(t, T=1000, cut_frac=0.1, ratio=32.0, eta_max=0.01):
    cut = math.floor(T * cut_frac)                       # = 100
    p = t / cut if t < cut else 1 - (t - cut) / (cut * (1 / cut_frac - 1))
    return eta_max * (1 + p * (ratio - 1)) / ratio
print("STLR  t=0   ->", round(stlr(0),   6))   # 0.000313  (the floor eta_max/ratio)
print("STLR  t=50  ->", round(stlr(50),  6))   # 0.005156
print("STLR  t=100 ->", round(stlr(100), 6))   # 0.010000  (the peak = eta_max)
print("STLR  t=200 ->", round(stlr(200), 6))   # 0.008924

# --- 0b. Worked example: discriminative per-layer rates, eta^{l-1} = eta^l / 2.6 ---
base = 0.01
print("disc lr per layer:", [round(base / 2.6**k, 6) for k in range(4)])
# [0.01, 0.003846, 0.001479, 0.000569]

# --- 1. A tiny synthetic "language": vocab V, two latent topics give text structure. ---
V, SEQ, NTOPIC = 20, 12, 2
g = torch.Generator().manual_seed(1)
topic_dist = torch.softmax(torch.randn(NTOPIC, V, generator=g) * 1.5, dim=1)
def gen_seq(topic):
    toks = [int(torch.randint(0, V, (1,), generator=g))]
    for _ in range(SEQ - 1):
        p = 0.5 * topic_dist[topic] + 0.5 * torch.softmax(torch.randn(V, generator=g), 0)
        toks.append(int(torch.multinomial(p, 1, generator=g)))
    return toks
def make(n):
    X, Y = [], []
    for _ in range(n):
        t = int(torch.randint(0, NTOPIC, (1,), generator=g)); Y.append(t); X.append(gen_seq(t))
    return torch.tensor(X), torch.tensor(Y)
Xun, _   = make(2000)   # unlabeled corpus for LM pretraining (lots)
Xtr, Ytr = make(20)     # FEW labeled examples
Xte, Yte = make(400)    # held-out test

# --- 2. Encoder (shared), LM head (pretraining), classifier head (concat pooling). ---
EMB, HID = 16, 24
class Encoder(nn.Module):
    def __init__(self):
        super().__init__(); self.emb = nn.Embedding(V, EMB); self.lstm = nn.LSTM(EMB, HID, batch_first=True)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x)); return h          # (B, SEQ, HID)
class LM(nn.Module):
    def __init__(self, enc): super().__init__(); self.enc = enc; self.dec = nn.Linear(HID, V)
    def forward(self, x): return self.dec(self.enc(x))
class Clf(nn.Module):
    def __init__(self, enc):
        super().__init__(); self.enc = enc
        self.head = nn.Sequential(nn.Linear(HID * 3, HID), nn.ReLU(), nn.Linear(HID, NTOPIC))
    def forward(self, x):
        h = self.enc(x)
        last, mx, mn = h[:, -1], h.max(1).values, h.mean(1)   # concat pooling (last + max + mean)
        return self.head(torch.cat([last, mx, mn], 1))

# --- 3. Stage 1: pretrain the LM (next-token prediction) on the unlabeled corpus. ---
def pretrain_lm(enc, epochs=8):
    lm = LM(enc); opt = torch.optim.Adam(lm.parameters(), lr=3e-3); lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(Xun))
        for i in range(0, len(Xun), 128):
            b = Xun[perm[i:i + 128]]
            opt.zero_grad(); loss = lf(lm(b[:, :-1]).reshape(-1, V), b[:, 1:].reshape(-1))
            loss.backward(); opt.step()

def accuracy(clf):
    clf.eval()
    with torch.no_grad(): return (clf(Xte).argmax(1) == Yte).float().mean().item()

# --- 4. Stage 3: fine-tune the classifier. tricks=True -> the three ULMFiT tricks. ---
def finetune(clf, tricks, epochs=40):
    enc_p, head_p = list(clf.enc.parameters()), list(clf.head.parameters())
    base = 0.01
    if tricks:   # discriminative: head at base, encoder (lower) at base/2.6
        opt = torch.optim.Adam([{'params': head_p, 'lr': base}, {'params': enc_p, 'lr': base / 2.6}])
    else:        # baseline: one global rate, no unfreezing
        opt = torch.optim.Adam([{'params': head_p + enc_p, 'lr': base}])
    lf = nn.CrossEntropyLoss(); accs = []
    for ep in range(epochs):
        if tricks:
            for pr in enc_p: pr.requires_grad_(ep >= epochs // 3)     # gradual unfreezing
            lr = stlr(ep, T=epochs)                                   # STLR schedule (Eqn 3)
            opt.param_groups[0]['lr'] = lr; opt.param_groups[1]['lr'] = lr / 2.6
        clf.train(); opt.zero_grad(); loss = lf(clf(Xtr), Ytr); loss.backward(); opt.step()
        for pr in clf.parameters(): pr.requires_grad_(True)
        accs.append(accuracy(clf))
    return accs

# (A) ULMFiT: pretrain -> transfer -> fine-tune with the three tricks
enc1 = Encoder(); pretrain_lm(enc1); acc_ulmfit = finetune(Clf(enc1), tricks=True)
# (B) From scratch: random encoder, plain fine-tune
torch.manual_seed(0); acc_scratch = finetune(Clf(Encoder()), tricks=False)
# (C) Ablation: pretrained encoder, tricks OFF
enc3 = Encoder(); pretrain_lm(enc3); acc_noTricks = finetune(Clf(enc3), tricks=False)

print("\nfinal test acc  (20 labels):")
print("  ULMFiT (pretrain + 3 tricks):", round(acc_ulmfit[-1], 3))
print("  from scratch                :", round(acc_scratch[-1], 3))
print("  ablation (pretrain, no tricks):", round(acc_noTricks[-1], 3))
# Our small run, not the paper's numbers. Typically: ULMFiT > ablation > from-scratch,
# i.e. pretraining helps AND the three tricks add a further margin.

## Visualize the data & results

_With only 20 labeled examples, does ULMFiT (pretrain + 3 tricks) beat from-scratch, and does the ablation (pretrain, no tricks) land in between?_

In [ ]:
import torch, torch.nn as nn, numpy as np, math
torch.manual_seed(0); np.random.seed(0)

V, SEQ, NTOPIC, EMB, HID = 20, 12, 2, 16, 24
g = torch.Generator().manual_seed(1)
topic_dist = torch.softmax(torch.randn(NTOPIC, V, generator=g) * 1.5, dim=1)
def gen_seq(t):
    s = [int(torch.randint(0, V, (1,), generator=g))]
    for _ in range(SEQ - 1):
        p = 0.5 * topic_dist[t] + 0.5 * torch.softmax(torch.randn(V, generator=g), 0)
        s.append(int(torch.multinomial(p, 1, generator=g)))
    return s
def make(n):
    X, Y = [], []
    for _ in range(n):
        t = int(torch.randint(0, NTOPIC, (1,), generator=g)); Y.append(t); X.append(gen_seq(t))
    return torch.tensor(X), torch.tensor(Y)
Xun, _ = make(2000); Xtr, Ytr = make(20); Xte, Yte = make(400)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__(); self.emb = nn.Embedding(V, EMB); self.lstm = nn.LSTM(EMB, HID, batch_first=True)
    def forward(self, x): h, _ = self.lstm(self.emb(x)); return h
class LM(nn.Module):
    def __init__(self, e): super().__init__(); self.enc = e; self.dec = nn.Linear(HID, V)
    def forward(self, x): return self.dec(self.enc(x))
class Clf(nn.Module):
    def __init__(self, e):
        super().__init__(); self.enc = e
        self.head = nn.Sequential(nn.Linear(HID * 3, HID), nn.ReLU(), nn.Linear(HID, NTOPIC))
    def forward(self, x):
        h = self.enc(x)
        return self.head(torch.cat([h[:, -1], h.max(1).values, h.mean(1)], 1))   # concat pooling

def pretrain(e, ep=8):
    lm = LM(e); opt = torch.optim.Adam(lm.parameters(), lr=3e-3); lf = nn.CrossEntropyLoss()
    for _ in range(ep):
        perm = torch.randperm(len(Xun))
        for i in range(0, len(Xun), 128):
            b = Xun[perm[i:i + 128]]
            opt.zero_grad(); lf(lm(b[:, :-1]).reshape(-1, V), b[:, 1:].reshape(-1)).backward(); opt.step()
def stlr(t, T, cf=0.1, r=32.0, em=0.01):
    cut = math.floor(T * cf)
    p = t / cut if t < cut else 1 - (t - cut) / (cut * (1 / cf - 1))
    return em * (1 + p * (r - 1)) / r
def acc(c):
    c.eval()
    with torch.no_grad(): return (c(Xte).argmax(1) == Yte).float().mean().item()
def finetune(c, tricks, ep=40):
    encp, hp, base = list(c.enc.parameters()), list(c.head.parameters()), 0.01
    opt = (torch.optim.Adam([{'params': hp, 'lr': base}, {'params': encp, 'lr': base / 2.6}])
           if tricks else torch.optim.Adam([{'params': hp + encp, 'lr': base}]))
    lf = nn.CrossEntropyLoss(); out = []
    for e in range(ep):
        if tricks:
            for pr in encp: pr.requires_grad_(e >= ep // 3)        # gradual unfreezing
            lr = stlr(e, ep); opt.param_groups[0]['lr'] = lr; opt.param_groups[1]['lr'] = lr / 2.6
        c.train(); opt.zero_grad(); lf(c(Xtr), Ytr).backward(); opt.step()
        for pr in c.parameters(): pr.requires_grad_(True)
        out.append(acc(c))
    return out

e1 = Encoder(); pretrain(e1); ulmfit = finetune(Clf(e1), True)
torch.manual_seed(0); scratch = finetune(Clf(Encoder()), False)
e3 = Encoder(); pretrain(e3); ablate = finetune(Clf(e3), False)
idx = np.linspace(0, 39, 20).astype(int)
print("ULMFiT :", [[int(i), round(ulmfit[i], 3)] for i in idx])
print("Ablation:", [[int(i), round(ablate[i], 3)] for i in idx])
print("Scratch:", [[int(i), round(scratch[i], 3)] for i in idx])
# ULMFiT ~0.94, ablation ~0.86, from-scratch ~0.80 (our small run, not the paper's numbers).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. ULMFiT (pretrained encoder + the three tricks) beat the from-scratch baseline
            on the tiny labeled set. Now run the middle case: take the pretrained encoder but fine-tune
            it with the tricks OFF (one global learning rate, no gradual unfreezing). Where does its accuracy
            land relative to full ULMFiT and to from-scratch, and what does each gap tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep the pretrained encoder; replace the per-layer/STLR/unfreeze schedule with a single constant learning rate and all layers trainable from step 0. Hold the labeled set, head, and seed fixed. — _Changing only the fine-tuning recipe (not the pretraining) isolates what the three tricks contribute on top of pretraining._
- Compare the three test accuracies: full ULMFiT, this pretrain-no-tricks ablation, and from-scratch. — _The gap (from-scratch &rarr; ablation) measures pretraining alone; the gap (ablation &rarr; full ULMFiT) measures the tricks on top._
- Note that the ablation lands BETWEEN the two: above from-scratch, below full ULMFiT. — _Pretraining already helps a lot; the tricks add a further margin by stopping the fine-tune from partly forgetting it._

**Answer:** The ablation sits in between &mdash; in our run, full ULMFiT &asymp; 0.94, the
                 pretrain-no-tricks ablation &asymp; 0.86, and from-scratch &asymp; 0.80 (our small run, not the
                 paper's numbers). The jump from 0.80 to 0.86 is what pretraining buys; the further jump
                 from 0.86 to 0.94 is what the three tricks buy by protecting that pretrained knowledge
                 during fine-tuning. Both ingredients contribute, which is exactly the paper's claim.

</details>

**Problem 2.** For an STLR run with $T = 2000$, $\mathit{cut\_frac}=0.1$, $\mathit{ratio}=32$,
            $\eta_{max}=0.01$, compute (a) the cut iteration, (b) the learning rate at $t=0$, and (c) the
            learning rate at the peak.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Cut: $\mathit{cut} = \lfloor 2000\cdot 0.1\rfloor = 200$. — _$\mathit{cut}$ is the floor of $T\cdot\mathit{cut\_frac}$ &mdash; the step where the ramp turns into decay._
- At $t=0$: $p=0$, so $\eta_0 = 0.01\cdot(1+0)/32 = 0.01/32 \approx 0.000313$. — _At $p=0$ the schedule sits at its floor $\eta_{max}/\mathit{ratio}$._
- At $t=\mathit{cut}=200$: $p=1$, so $\eta = 0.01\cdot(1+31)/32 = 0.01$. — _At $p=1$ the numerator is $1+(\mathit{ratio}-1)=\mathit{ratio}$, cancelling the denominator &mdash; the rate equals $\eta_{max}$._

**Answer:** (a) $\mathit{cut} = 200$. (b) $\eta_0 = 0.01/32 \approx \mathbf{0.000313}$, the floor. (c) at
                 the peak $t=200$, $\eta = \mathbf{0.01} = \eta_{max}$. The shape is identical to the
                 $T=1000$ worked example &mdash; only the cut point scales with $T$.

</details>

**Problem 3.** You are fine-tuning a 5-layer pretrained encoder with discriminative rates. You set the top
            layer's rate to $\eta^L = 0.01$ and use $\eta^{l-1}=\eta^l/2.6$. What rate does the
            bottom (5th-from-top) layer get, and why does the paper want it so small?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the factor four times (top to bottom is four steps down): $0.01 / 2.6^4$. — _Each layer down divides by 2.6; the bottom layer is 4 layers below the top._
- Compute $2.6^4 \approx 45.7$, so the bottom rate $\approx 0.01/45.7 \approx 0.000219$. — _The geometric decay makes the lowest layer move far slower than the top._
- Recall what the bottom layers hold: general, broadly-useful features from pretraining. — _Those should be preserved, not overwritten, so the tiny rate keeps them nearly fixed while the top adapts._

**Answer:** The bottom layer's rate is $0.01/2.6^4 \approx \mathbf{0.000219}$ &mdash; roughly
                 $46\times$ smaller than the top. The paper wants it tiny because the lowest layers carry the
                 most general, transferable knowledge from pretraining; moving them fast would cause
                 catastrophic forgetting, so they are nearly frozen while the task-specific top layers do the
                 adapting.

</details>